In [50]:
import pandas as pd
import numpy as np

In [84]:
cd=pd.read_csv("cars.csv")

In [91]:
cd.to_csv("cars.csv",index=False)

In [29]:
cd.isnull().sum()

baslik                     0
konum                      0
fiyat                      0
ilan_tarihi                0
marka                      0
seri                       0
model                      0
yil                        0
kilometre                  0
vites_tipi                 0
yakit_tipi                 0
kasa_tipi                  0
renk                       0
motor_hacmi                0
motor_gucu                 0
cekis                      0
arac_durumu                0
ortalama_yakit_tuketimi    0
yakit_deposu               0
boya_degisen               0
kimden                     0
tramer                     0
dtype: int64

In [31]:
cd = cd.drop('baslik', axis=1)
cd = cd.drop('renk', axis=1)
cd = cd.drop('arac_durumu', axis=1)
cd = cd.drop('kimden', axis=1)
cd = cd.drop('boya_degisen', axis=1)
cd = cd.drop('takasa_uygun', axis=1)
cd = cd.drop('ilan_tarihi', axis=1)

In [85]:
cd.head()

,konum,fiyat,marka,seri,model,yil,kilometre,vites_tipi,yakit_tipi,kasa_tipi,motor_hacmi,motor_gucu,cekis,ortalama_yakit_tuketimi,yakit_deposu,tramer,degisen,boyali
0,Adana,1169000.0,Renault,Megane,1.3 TCe Joy Comfort,2022,124000.0,Otomatik,Benzin,Sedan,1201.0,126.0,Önden Çekiş,5.3,50.0,71300.0,1,0
1,Adana,1520000.0,Skoda,Octavia,1.0 e-Tec Elite,2022,99000.0,Otomatik,Hibrit,Sedan,1201.0,101.0,Önden Çekiş,4.3,48.0,56060.0,0,1
2,Adana,469999.0,Peugeot,207,1.4 Trendy,2009,207500.0,Düz,LPG & Benzin,Hatchback/5,1360.0,91.0,Önden Çekiş,6.3,50.0,0.0,0,0
3,Adana,1150000.0,Audi,A4,A4 Sedan 2.0 TDI Quattro,2013,219000.0,Otomatik,Dizel,Sedan,1801.0,176.0,4WD (Sürekli),5.1,63.0,0.0,1,4
4,Adana,1430000.0,Toyota,Corolla,1.5 Dream,2021,63000.0,Otomatik,Benzin,Sedan,1490.0,125.0,Önden Çekiş,5.8,50.0,0.0,0,0


In [12]:
cd["konum"] = cd["konum"].str.split(",",expand=True)[1]

In [16]:
cd['yakit_deposu'] = cd['yakit_deposu'].fillna(
    cd.groupby(['seri','model'])['yakit_deposu'].transform('first'))

In [17]:
cd['ortalama_yakit_tuketimi'] = cd['ortalama_yakit_tuketimi'].fillna(
    cd.groupby(['seri','model'])['ortalama_yakit_tuketimi'].transform('first'))

In [25]:
cd=cd[~cd["yakit_deposu"].isnull()]
cd = cd[~cd["arac_durumu"].isnull()]
cd = cd[~cd["model"].isnull()]
cd = cd[~cd["kilometre"].isnull()]
cd = cd[~cd["renk"].isnull()]
cd = cd[~cd["motor_hacmi"].isnull()]
cd = cd[~cd["arac_durumu"].isnull()]
cd = cd[~cd["motor_gucu"].isnull()]
cd = cd[~cd["ortalama_yakit_tuketimi"].isnull()]
cd = cd[~cd["cekis"].isnull()]

In [28]:
cd["tramer"] = cd["tramer"].fillna(0)

In [14]:
cd['fiyat'] = cd['fiyat'].str.replace(' TL', '', regex=False).str.replace('.', '', regex=False).astype(float)

cd['kilometre'] = cd['kilometre'].str.replace(' km', '', regex=False).str.replace('.', '', regex=False).astype(float)

cd['ortalama_yakit_tuketimi'] = cd['ortalama_yakit_tuketimi'].str.replace(' lt', '', regex=False).str.replace(',', '.', regex=False).astype(float)

cd['yakit_deposu'] = cd['yakit_deposu'].str.replace(' lt', '', regex=False).astype(float)

In [23]:
cd['motor_gucu'] = cd['motor_gucu'].str.replace(' HP', '', regex=False)
cd['motor_gucu'] = cd['motor_gucu'].str.replace(' hp', '', regex=False)
cd['motor_hacmi'] = cd['motor_hacmi'].str.replace(' cm3', '', regex=False)
cd['motor_hacmi'] = cd['motor_hacmi'].str.replace(' cc', '', regex=False)

In [35]:
cd["motor_hacmi"] = cd["motor_hacmi"].str.replace("' e kadar","")

In [44]:
cd["motor_hacmi"] = cd["motor_hacmi"].str.split("-").str[0]

In [46]:
cd["motor_gucu"] = cd["motor_gucu"].str.split("-").str[0]

In [51]:
cd["degisen"] = 0
cd["boyali"] = 0

In [53]:
mask = cd["boya_degisen"].str.contains("değişen|boyalı")

In [57]:
cd.loc[mask, "degisen"] = cd.loc[mask, "boya_degisen"].str.extract(r"(\d+)\s*değişen")[0].fillna(0).astype(int)
cd.loc[mask, "boyali"] = cd.loc[mask, "boya_degisen"].str.extract(r"(\d+)\s*boyalı")[0].fillna(0).astype(int)

In [78]:
cd.dtypes

konum                       object
fiyat                      float64
ilan_tarihi                 object
marka                       object
seri                        object
model                       object
yil                          int64
kilometre                  float64
vites_tipi                  object
yakit_tipi                  object
kasa_tipi                   object
motor_hacmi                float64
motor_gucu                 float64
cekis                       object
ortalama_yakit_tuketimi    float64
yakit_deposu               float64
tramer                     float64
degisen                      int64
boyali                       int64
dtype: object

In [75]:
cd["motor_gucu"] = cd["motor_gucu"].str.replace(" ve üzeri","")

In [76]:
cd["motor_gucu"] = cd["motor_gucu"].astype(float)

In [89]:
cd = cd[cd["fiyat"] < cd["fiyat"].quantile(0.99)]